# Boosting

In this lesson, we introduce **Boosting**, a powerful ensemble learning technique that builds models **sequentially** to improve predictive performance.

Just like in Bagging, Boosting is based on combining multiple models — but the way it does this is fundamentally different.

- In **Bagging**, individual models are trained independently on **bootstrap samples drawn with replacement** from the training data.
- In **Boosting**, models are built **sequentially**, where each new model is trained to **correct the errors** made by the previous ones.

This results in a model that increasingly focuses on the hard-to-predict examples, which may improve accuracy over time.

By the end of this lesson, you should be able to:

- Understand how Boosting works and how it differs from Bagging.
- Explain the idea behind **Adaptive Boosting (AdaBoost)**.
- Understand how Boosting can be combined with **gradient descent**, leading to **Gradient Boosting** methods.

## 1. From Trees to Boosting

Boosting usually employs very simple base models with slightly better performance than random guessing, which is why they are usually referred to as **weak learners**. An example would be a decision tree with only one split.

The original boosting procedure can be summarized as follows:

1. Draw a random subset (sample) of training examples, $d_1$ , without replacement from the training dataset, to fit a weak learner, $C_1$.
2. Draw a second training subset, $d_2$, with half its examples correctly classified by $C_1$ and half misclassified by $C_1$, to fit another weak learner, $C_2$. 
3. Find the training examples on which $C_1$ and $C_2$ disagree, forming a subset $d_3$, to fit a third weak learner, $C_3$.
4. Combine the weak learners $C_1$, $C_2$, and $C_3$ via majority voting.

Boosting can lead to a decrease in bias as well as variance compared to bagging models. As a result, it is not surprising to see algorithms under this procedure scoring at the top in many benchmarks.

## 2. Adaptive Boosting (AdaBoost)

In contrast to the original boosting procedure described before, Adaptive boosting (AdaBoost) uses the complete training data to fit the weak learners, such that in each iteration the training examples are reweighted to build a strong model that learns from the mistakes of the previous weak learners in the ensemble.

The final predictions are then made taking into account all the different iterations.

The original AdaBoost algorithm (Freund & Schapire, 1997) is as follows:

Given:

- A training dataset $X_{ij}$ with labels $y_i \in \{-1, +1\}$ 
- A base learning algorithm  
- A number of boosts $M$


1. Initialize sample weights uniformly:

$$w_i^{(1)} = \frac{1}{n}, \quad \text{for } i = 1, \ldots, n$$

2. For $m = 1,\ldots,M$:

    1. Fit a weak learner: $C_m = \mbox{train}(X,y, w^{(m)})$
    2. Predict class labels: $\hat{y}_i^{(m)} = C_m(x_i), \qquad i=1,\ldots,n$
    3. Compute weighted error rate:
        $$\varepsilon_m = \sum_{i=1}^n w_i^{(m)} \cdot \mathbf{1}(\hat{y}_i^{(m)}\neq y_i)$$
   4. Compute the coefficient:
        $$\alpha_m = \frac{1}{2} \log\left(\frac{1 - \varepsilon_m}{\varepsilon_m}\right)$$
   5. Update weights:
        $$w_i^{(m+1)} = w_i^{(m)} \cdot \exp\left(-\alpha_m \times y_i \times \hat{y}_i^{(m)}\right)$$
   6. Normalize the weights:
        $$w_i^{(m+1)} \leftarrow \frac{w_i^{(m+1)}}{\sum_{j=1}^n w_j^{(m+1)}}, \qquad i=1,\ldots,n$$



4. The final prediction is a weighted majority vote:

$$\hat{y}(x) = \begin{cases} +1, & \displaystyle\sum_{m=1}^M \alpha_m C_m(x)>0, \\ -1, & \text{otherwise}. \end{cases}$$



Let's look more closely at how AdaBoost works.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.utils import resample
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.inspection import DecisionBoundaryDisplay

import matplotlib.pyplot as plt

X, y = make_classification(n_samples=500, n_features=2, n_informative=2, n_redundant=0, random_state=42, class_sep=0.15, flip_y=0.1)

X_train, X_report, y_train, y_report = train_test_split(X, y, test_size=0.2, random_state=1234)

In [ ]:
# HERE IT COMES CODE: Import DecisionTreeClassifier

tree = None # HERE IT COMES CODE: create a decision tree with maximum depth 1 in its splitting with random_state=1234
# HERE IT COMES CODE: Fit to training

# HERE IT COMES CODE: calculate accuracy in reporting set 

In [ ]:
# HERE IT COMES CODE: Import confusion_matrix and ConfusionMatrixDisplay
cm = None # HERE IT COMES CODE: Calculate confusion matrix in reporting set
# HERE IT COMES CODE: Plot it using ConfusionMatrixDisplay

In [ ]:
fig, ax = plt.subplots(figsize=(5,5))

DecisionBoundaryDisplay.from_estimator(tree, 
                                       X_train, 
                                       cmap=plt.cm.RdBu, 
                                       response_method="predict", 
                                       ax=ax)

ax.scatter(X_report[y_report==0][:,0], X_report[y_report==0][:,1], c="r", edgecolor="black", s=50)
ax.scatter(X_report[y_report==1][:,0], X_report[y_report==1][:,1], c="b", edgecolor="black", s=50)
ax.set_title(f"Decision Tree \n (acc = {tree.score(X_report, y_report)*100:.1f})", size=25)
ax.set_xlabel("feature 1", size=20)
ax.set_ylabel("feature 2", size=20)
plt.tight_layout()

The decision tree has maximum depth 2 in its splitting, so this classifier in general will not perform the best. Let's apply the AdaBoost recipe to try to improve it.

First, initialize sample weights uniformly:

In [ ]:
weights = np.array([1/len(y_train)]*len(y_train))
weights[:10]

Now, let's suppose we just do 1 boost

In [ ]:
# 2. A Fit the same weak learner with the above weights using sample_weight 
# HERE COMES CODE

# 2. B Predict class labels 
y_pred_train = None # HERE COMES CODE

# 2. C Compute weighted error rate
epsilon = None # HERE COMES CODE

# 2. D Compute the learner coefficient
alpha = None # HERE COMES CODE

# 2. E Update weights
new_weights = None # HERE COMES CODE: pay attention to how we implement this...

# 2. F Normalize weights
new_weights = None # HERE COMES CODE

Let's look at how weights change after this boost.

In [ ]:
print("Well-classified example: before boost, w =", 
      weights[y_pred_train==y_train][0], 
      "after boost, w =", 
      new_weights[y_pred_train==y_train][0])

print("Misclassified example: before boost, w =", 
      weights[y_pred_train!=y_train][0], 
      "after boost, w =", 
      new_weights[y_pred_train!=y_train][0])


As you can see, those misclassified get a larger weight. And that's good, because that means the next classifier will try to focus more on these particular examples to try to classify them well.

Let's now iterate for 100 boosts, for example:

In [ ]:
n_boost = 100
weights = np.ones(len(y_train)) / len(y_train)

learners = []
alphas = []

for ii in range(n_boost):
    clf = DecisionTreeClassifier(max_depth=1, random_state=1234)

    # 2. A Fit weak learner
    clf.fit(X_train, y_train, sample_weight=weights)
    
    # 2. B Predict class labels 
    y_pred_train = clf.predict(X_train)
    
    # 2. C Compute weighted error rate
    epsilon = weights.dot(y_pred_train!= y_train) 
    
    # 2. D Compute the learner coefficient
    alpha = 0.5 * np.log((1 - epsilon) / (epsilon + 1e-10))
    
    # 2. E Update weights
    weights = weights* np.exp(alpha * (y_pred_train!= y_train) * 2 - alpha)
    
    # 2. F Normalize weights
    weights /= weights.sum()

    learners.append(clf)
    alphas.append(alpha)



In [ ]:
final_train = 0
final_report = 0
for clf, alpha in zip(learners, alphas):
    y_pred_train = 2*clf.predict(X_train) -1 # as {-1, +1}
    y_pred_report = 2*clf.predict(X_report) - 1 # as {-1, +1}

    final_train += alpha*y_pred_train
    final_report += alpha*y_pred_report

final_train = (final_train >0).astype(int)
final_report = (final_report>0).astype(int)

The final labels are determined by a weighted majority vote:

In [ ]:
print(accuracy_score(y_train, final_train))
print(accuracy_score(y_report, final_report))

So far, we have described AdaBoost in its original form, where each weak learner's contribution to the final prediction is weighted by a factor $\alpha_m$ determined entirely by its performance. However, modern implementations of AdaBoost typically introduce a learning rate — a hyperparameter that scales these $\alpha_m$ values.

The learning rate controls how much each individual learner contributes to the final model. This idea, known as shrinkage, helps improve generalization by preventing overconfident updates and reducing the risk of overfitting. Although it slows down the learning process, it often results in better performance, especially when combined with a larger number of estimators.

### 2.1 Implementation in scikit-learn

`scikit-learn` provides convenient implementations of AdaBoost through the `ensemble.AdaBoostClassifier` and `ensemble.AdaBoostRegressor` classes, which can be used for classification and regression tasks, respectively. 

<div class="alert alert-info"> 

𝐈𝐌𝐏𝐎𝐑𝐓𝐀𝐍𝐓

Modern implementations of AdaBoost typically introduce a **learning rate**, a hyperparameter that scales the learner weights $\alpha$.

The learning rate controls how much each individual learner contributes to the final model. For example, a higher learning rate increases the impact of each classifier. While this can improve performance, it also increases the risk of overfitting — so tuning this parameter is essential.

`scikit-learn` includes this option in both `ensemble.AdaBoostClassifier` and `ensemble.AdaBoostRegressor`.

</div>



Let's reproduce what we saw before, but now with this implementation.

In [ ]:
# HERE COMES CODE: Import AdaBoostClassifier

ada = None # HERE COMES CODE: Define object AdaBoostClassifier with a decision tree with maximum depth 1 in its splitting, 100 estimators, learning_rate=1, random_state=10

# HERE COMES CODE: Fit to the training

# HERE COMES CODE: Score on the reporting set

Let's compare AdaBoost’s decision boundary with that of the base decision tree:

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=(10,7), sharey=True)

for ii, (case, estim) in enumerate(zip(["Decision Tree","AdaBoost"], 
                                       [tree, ada])):
    DecisionBoundaryDisplay.from_estimator(estim, 
                                           X_train, 
                                           cmap=plt.cm.RdBu, 
                                           response_method="predict", 
                                           ax=axs[ii])

    axs[ii].scatter(X_report[y_report==0][:,0], X_report[y_report==0][:,1], c="r", edgecolor="black", s=50)
    axs[ii].scatter(X_report[y_report==1][:,0], X_report[y_report==1][:,1], c="b", edgecolor="black", s=50)
    axs[ii].set_title(f"{case} \n (acc = {estim.score(X_report, y_report)*100:.1f})", size=25)
    axs[ii].set_xlabel("feature 1", size=20)
    axs[ii].set_ylabel("feature 2", size=20)
plt.tight_layout()

**We can see that we gain a substantial improvement and furthermore, AdaBoost introduces additional flexibility to the decision boundaries, resulting in better classification of some observations.**

### 2.2 Pause and think exercise: Boosting with Logistic Regression

Repeat the experiment using `LogisticRegression` as the base learner, instead of the decision tree. Compare this to using a `LogisticRegression` alone.

After this, respond to:

- How does the performance of boosted logistic regression compare to a single logistic regression model?
- Why do you think boosting does not work here as effectively?

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.linear_model import LogisticRegression

# TODO: 
# 1- Create a logistic regression object
# 2- Fit this object to the training set
# 3- Calculate accuracy of the model on the reporting set
logReg = None 

# TODO: 
# 1- Create an AdaBoost classifier with Logistic Regression as the base learner. Use 100 of these models trained sequentially. Assume learning_rate=1 and algorithm='SAMME'.
# 2- Fit this object to the training set
# 3- Calculate accuracy of the model on the reporting set
ada_log = None


## 3. Gradient Boosting: Boosting + Gradient Descent

Gradient Boosting combines ideas from **Boosting** and **Gradient Descent**.

Instead of reweighting samples like AdaBoost, Gradient Boosting fits each new model to the **negative gradients of the chosen cost function**. For squared-error cost, these are the **residuals** of the current ensemble.

We are not going to get into the math here (refer to Chapter 7 for that), but the key idea is:

- Start with a simple model that makes an initial prediction.
- At each step, compute how the current model is wrong (residuals or gradients of the cost).
- Fit a new weak learner (usually a shallow decision tree) to these negative gradients.
- Add the predictions of this learner to the existing model, scaled by a **learning rate**.

As a result, the optimization is guided by **cost gradients**, not by weighted classification errors.

This process is repeated for many iterations, gradually improving the model’s performance by reducing the cost.

Gradient Boosting works with **any differentiable cost function** (e.g., squared error for regression, log-loss for classification).

Gradient Boosting supports different cost functions and offers flexibility in model design. Its performance and tendency to overfit depend on settings such as tree depth, learning rate, and the number of boosting iterations.


### 3.1 Implementation in scikit-learn

`scikit-learn` provides easy-to-use implementations of Gradient Boosting through `sklearn.ensemble.GradientBoostingRegressor` and `sklearn.ensemble.GradientBoostingClassifier`.

These models follow the principles described above: they sequentially add shallow trees to correct the errors of the current ensemble, using gradient descent to minimize a specified cost function.


In [ ]:
# HERE COMES CODE: Import GradientBoostingClassifier
# Fit gradient boosting model
gb = None # HERE COMES CODE: Fit GradientBoostingClassifier with n_estimators=100, learning_rate=1, max_depth=1, random_state=1234
# HERE COMES CODE: Fit the model to the training data

# Evaluate
# HERE COMES CODE: Calculate accuracy score in the reporting set

And let's compare the decision boundaries:

In [ ]:
fig, axs = plt.subplots(ncols=3, figsize=(15,7), sharey=True)

for ii, (case, estim) in enumerate(zip(["Decision Tree","AdaBoost", "Gradient Boosting"], 
                                       [tree, ada, gb])):
    DecisionBoundaryDisplay.from_estimator(estim, 
                                           X_train, 
                                           cmap=plt.cm.RdBu, 
                                           response_method="predict", 
                                           ax=axs[ii])

    axs[ii].scatter(X_report[y_report==0][:,0], X_report[y_report==0][:,1], c="r", edgecolor="black", s=50)
    axs[ii].scatter(X_report[y_report==1][:,0], X_report[y_report==1][:,1], c="b", edgecolor="black", s=50)
    axs[ii].set_title(f"{case} \n (acc = {estim.score(X_report, y_report)*100:.1f})", size=25)
    axs[ii].set_xlabel("feature 1", size=20)
    axs[ii].set_ylabel("feature 2", size=20)

plt.tight_layout()

## 4. Explainability

For decision trees, feature importance scores summarize how much each feature helps reduce impurity across the splits.

Both AdaBoost and Gradient Boosting in `scikit-learn` provide a `feature_importances_` attribute when using decision trees. AdaBoost combines the trees’ importance scores using the learners’ weights, while Gradient Boosting aggregates impurity reductions across its trees.

In [ ]:
ada.feature_importances_

In [ ]:
gb.feature_importances_

## 5. Summary

- **Boosting** is an ensemble technique that combines weak learners in a **sequential** manner, where each new model focuses on the **errors** made by the previous ones.
- In **Adaptive Boosting (AdaBoost)**,  models are trained with **weighted** data to emphasize previously misclassified examples.
- **Gradient Boosting**, instead, generalizes this idea by using gradient descent to iteratively minimize a chosen cost function.

## 📝 6. Exercises

### 6.1 Exercise  – AdaBoost for Regression

In this exercise you will be using `AdaBoostRegressor` to solve a simple regression problem, and manually verify that the final prediction is a **weighted median** of the weak learners' predictions.

In order, do the following:

1. Generate a toy dataset. You can use the function `make_regression` from `scikit-learn` for this.

2. Split it into training and reporting sets.

3. Use `AdaBoostRegressor` to fit the training set. Use a decision tree with maximum depth 1 in its splitting as the base model, and at least 100 boosts.

4. Create a scatter plot showing the predictions vs observed values in the reporting set. Also include the model’s performance in the plot title.

5. Replicate the predictions in 4 manually. For this, use each base learner and weight from the `.estimators_` and `.estimator_weights_` attributes after fitting the AdaBoost, and compute the final prediction as the **weighted median** of the individual predictions.

In [ ]:
#YOUR CODE

### 6.2 Exercise – Gradient Boosting with Learning Rate Tuning

In this exercise, you will explore how the learning rate affects Gradient Boosting performance.

Using the same toy data as the one used throughout this tutorial:

Split the training data into fitting and tuning sets. Use these sets for the comparisons below, keeping the reporting set aside.

1. Calculate the fitting accuracy and tuning accuracy for a varying range of learning rates.  You may try this list of values: 0.001, 0.01, 0.1, 0.3, 0.5, and 1.0.

2. Plot these accuracies against the learning rates.  

3. Plot the decision boundaries for a couple of extreme cases (e.g., very low and very high learning rates).  You may use the same visualization function we have used above.

**Discussion**: How does performance change with the learning rate? Are fitting and tuning accuracies aligned? What happens to the decision boundary as the learning rate increases?

In [ ]:
#YOUR CODE